# 08 — Integrazione LLM + MCP per Movie RAG

Questo notebook verifica progressivamente la nuova integrazione senza modificare la collection Qdrant. Gli obiettivi sono:

1. verificare la chiave e una chiamata diretta a Gemini;
2. osservare l'output strutturato del **Planner**;
3. eseguire il flusso completo **Planner → MCP/Qdrant → Answer**;
4. ispezionare piano, filtri, risultati, trace e risposta grounded;
5. mostrare come cambiare provider/modello o abilitare il Critic senza cambiare il contratto MCP.

> La configurazione predefinita usa `dense_v1`, cioè il retriever risultato migliore nei test precedenti. Il reranker resta un profilo sperimentale del server MCP e non viene scelto dalla LLM.

## 1. Architettura e responsabilità

Gli agenti sono ruoli logici, non processi autonomi. Ogni `run(...)` è una chiamata LLM con prompt e schema versionati. L'orchestratore Python legge l'oggetto restituito e decide il passo successivo.

| Livello | Responsabilità |
|---|---|
| Planner LLM | Estrae query semantica, lingua, filtri e ambiguità |
| Orchestratore Python | Mantiene lo stato, applica i rami e limita i retry |
| MCP | Espone `search_movies` e `get_movie_details` |
| Qdrant/BGE-M3 | Recupera i film del catalogo |
| Answer LLM | Formula la risposta usando esclusivamente gli output MCP |

Il Critic è già disponibile, ma rimane disabilitato nel test principale per confrontare prima una pipeline semplice e riproducibile.

## 2. Installazione

Dalla root del progetto, con il virtual environment attivo:

```bash
python -m pip install -e ".[bge,mcp,llm-gemini,test]"
```

Per installare anche l'adapter OpenAI usare `.[bge,mcp,llm,test]`. Prima di proseguire devono essere attivi Qdrant e la collection popolata dal notebook 05.

In [1]:
#cella, Jupyter controllerà automaticamente se i file sorgente dei moduli importati sono cambiati
# e, in caso, li ricaricherà da solo prima di lanciare il codice
%load_ext autoreload
%autoreload 2

In [25]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Root del progetto non trovata: manca pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/danieleraveglia/Documents/uniMi/TESI/progetto


## 3. Credenziali locali

Il file `.env` non viene caricato implicitamente dal package: è l'applicazione a scegliere esplicitamente il file. La cella seguente verifica soltanto la presenza di `GEMINI_API_KEY`; non ne mostra né salva il valore. `.env` è escluso da Git.

In [26]:
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH, override=False)

gemini_key_available = bool(os.getenv("GEMINI_API_KEY"))
OpenAI_key_available = bool(os.getenv("OPENAI_API_KEY"))

print({
    "env_file_found": ENV_PATH.exists(),
    "GEMINI_API_KEY_available": gemini_key_available,
    "OPENAI_API_KEY_available": OpenAI_key_available,
})
assert gemini_key_available, (
    "GEMINI_API_KEY non trovata. Inseriscila nel .env della root e "
    "riavvia il kernel, senza stamparla nel notebook."
)
assert OpenAI_key_available, (
    "OPENAI_API_KEY non trovata. Inseriscila nel .env della root e "
    "riavvia il kernel, senza stamparla nel notebook."
)

{'env_file_found': True, 'GEMINI_API_KEY_available': True, 'OPENAI_API_KEY_available': True}


## 4. Parametri espliciti dell'esperimento

Queste sono le sole scelte da cambiare per il test. I nomi devono esistere nel registry `configs/llm_profiles_v1.json`. La strategia `none` conserva la lingua originale della query; `translate_to_catalog_language` chiede invece al Planner di produrre la query nella lingua del catalogo. Le due strategie vanno valutate separatamente.

In [27]:
LLM_CONFIG_PATH = PROJECT_ROOT / "configs" / "llm_profiles_v1.json"
WORKFLOW_ID = "grounded_v1"

# Ogni ruolo può usare un profilo diverso, purché dichiarato nel JSON.
ROLE_PROFILE_OVERRIDES = {
    "planner": "openai_default",
    "answer": "openai_default",
}
# ROLE_PROFILE_OVERRIDES = {
#     "planner": "gemini_free",
#     "answer": "gemini_free",
# }

ENABLE_CRITIC = False
#strategia per eventuale rewrite della qury dell'utente prima di passarlo al retriever. Può essere "none" o "translate_to_catalog_language"
QUERY_REWRITE_STRATEGY = "none" #translate_to_catalog_language
USER_QUERY = (
    "Vorrei un film malinconico sulla memoria e l'identità, "
    "non horror e sotto le due ore."
)

In [28]:
from movie_rag import load_llm_config

catalog = load_llm_config(LLM_CONFIG_PATH)
workflow_profile = catalog.get_workflow(WORKFLOW_ID)

profile_rows = [
    {
        "profile_id": profile_id,
        "provider": profile.provider,
        "model": profile.model,
        "api_key_env": profile.api_key_env,
        "temperature": profile.temperature,
    }
    for profile_id, profile in catalog.llm_profiles.items()
]
display(pd.DataFrame(profile_rows))
display(pd.DataFrame([workflow_profile.model_dump()]))

,profile_id,provider,model,api_key_env,temperature
0,gemini_free,gemini,gemini-3.5-flash-lite,GEMINI_API_KEY,0.0
1,gemini_free_pro,gemini,gemini-3.6-flash,GEMINI_API_KEY,0.0
2,openai_default,openai,gpt-5.6-luna,OPENAI_API_KEY,0.0


,workflow_id,planner_agent,critic_agent,answer_agent,enable_critic,max_retries,search_limit,details_limit,catalog_language,query_rewrite_strategy
0,grounded_v1,planner,critic,answer,False,1,10,5,en,none


## 5. Smoke test API LLM

Questa chiamata non coinvolge MCP, Qdrant o BGE-M3. Serve a isolare errori di chiave, quota, modello o rete. La trace mostra metadati normalizzati e non contiene la chiave.

In [22]:
# verifica modelli disponibili per OpenAI
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(PROJECT_ROOT / ".env")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.getenv("OPENAI_BASE_URL") or None,
)

models = client.models.list()

available_models = sorted(model.id for model in models.data)

for model_id in available_models:
    print(model_id)

gpt-5.4-mini
gpt-5.4-nano
gpt-5.5
gpt-5.6-luna
gpt-5.6-terra


In [24]:
# GEMINI
from movie_rag import GeminiProvider

gemini_profile = catalog.get_llm_profile("gemini_free")
gemini_provider = GeminiProvider(gemini_profile)

smoke = await gemini_provider.generate_text(
    agent="smoke_test",
    prompt_version="smoke_v1",
    system_prompt="Reply with exactly: API_OK",
    user_prompt="Verify that this API call works.",
    temperature=0.0,
    max_output_tokens=500,
)
print(smoke.text)
display(pd.DataFrame([smoke.trace.model_dump(mode="json")]))

#OPENAI
from movie_rag import OpenAIProvider

openai_profile = catalog.get_llm_profile("openai_default")
openai_provider = OpenAIProvider(openai_profile)

smoke = await openai_provider.generate_text(
    agent="smoke_test",
    prompt_version="smoke_v1",
    system_prompt="Reply with exactly: API_OK",
    user_prompt="Verify that this API call works.",
    max_output_tokens=500,
)
print(smoke.text)
display(pd.DataFrame([smoke.trace.model_dump(mode="json")]))


API_OK


,agent,provider,profile_id,model,prompt_version,started_at,duration_ms,usage,request_id
0,smoke_test,gemini,gemini_free,gemini-3.5-flash-lite,smoke_v1,2026-09-22T10:24:23.698735Z,1130.824584,"{'input_tokens': 16, 'output_tokens': 3, 'tota...",WFeyav7cJYCJkdUPztHawAQ


API_OK


,agent,provider,profile_id,model,prompt_version,started_at,duration_ms,usage,request_id
0,smoke_test,openai,openai_default,gpt-5.6-luna,smoke_v1,2026-09-22T10:24:24.851197Z,1928.260833,"{'input_tokens': 23, 'output_tokens': 6, 'tota...",resp_0b759e714208145e006ab25759a6d487d2b275846...


## 6. Test isolato del Planner

Il Planner riceve la richiesta e restituisce un `RetrievalPlan` validato. Non può chiamare MCP e non può scegliere il retriever. Controlliamo esplicitamente quali parti diventano query semantica e quali diventano filtri esatti.

In [29]:
from movie_rag import (
    ConversationState,
    LLMProviderRegistry,
    build_agent_set,
)

provider_registry = LLMProviderRegistry(catalog)
effective_workflow_profile = workflow_profile.model_copy(
    update={
        "enable_critic": ENABLE_CRITIC,
        "query_rewrite_strategy": QUERY_REWRITE_STRATEGY,
    }
)
agents = build_agent_set(
    catalog=catalog,
    workflow=effective_workflow_profile,
    provider_registry=provider_registry,
    role_profile_overrides=ROLE_PROFILE_OVERRIDES,
)

planner_state = ConversationState(
    workflow_id=WORKFLOW_ID,
    original_query=USER_QUERY,
    catalog_language=workflow_profile.catalog_language,
    query_rewrite_strategy=QUERY_REWRITE_STRATEGY,
)
plan, planner_trace = await agents.planner.run(
    planner_state, effective_workflow_profile
)
display(plan.model_dump())
display(pd.DataFrame([planner_trace.model_dump(mode="json")]))

{'action': 'search',
 'detected_language': 'it',
 'semantic_query': "Un film malinconico sulla memoria e l'identità, con un tono riflessivo ed emotivo.",
 'semantic_query_language': 'it',
 'rewrite_strategy': 'none',
 'filters': {'min_runtime': None,
  'max_runtime': 120.0,
  'min_year': None,
  'max_year': None,
  'languages': [],
  'countries': [],
  'include_genres': [],
  'exclude_genres': ['horror'],
  'adult': None},
 'ambiguities': [],
 'clarification_question': None,
 'reason': 'La richiesta contiene preferenze tematiche chiare e vincoli supportati: niente horror e durata inferiore a due ore.'}

,agent,provider,profile_id,model,prompt_version,started_at,duration_ms,usage,request_id
0,planner,openai,openai_default,gpt-5.6-luna,planner_v1,2026-09-22T10:25:29.400160Z,4337.993375,"{'input_tokens': 870, 'output_tokens': 177, 't...",resp_0c77a7d03c1285f7006ab2579a6ed087d2a655514...


### Controllo manuale del piano

Prima di usare il risultato verificare:

- `semantic_query` contiene tema, tono e contenuto, non confronti numerici;
- `max_runtime=120` e `exclude_genres=["Horror"]` sono filtri;
- con strategia `none`, la query semantica resta in italiano;
- il Planner non ha inventato vincoli assenti dalla richiesta.

## 7. Flusso end-to-end con MCP in-process

Usiamo il server MCP reale ma nello stesso processo Python. È comunque un round-trip MCP: il gateway chiama i tool pubblici e ne valida l'output. Il registry Qdrant viene condiviso tra le prove, così BGE-M3 non viene ricaricato a ogni query.

Prerequisiti: Qdrant raggiungibile su `QDRANT_URL` (default `http://localhost:6333`) e profilo MCP `dense_v1`.

In [7]:
from mcp import Client

from movie_rag import (
    LLMProviderRegistry,
    MCPMovieToolGateway,
    MovieSearchServiceRegistry,
    build_movie_assistant_workflow,
)
from movie_rag.mcp_server import create_mcp_server

# Le scelte tecniche del retrieval restano configurazione del server MCP.
os.environ.setdefault("MOVIE_RAG_PROFILE", "dense_v1")
os.environ.setdefault("QDRANT_URL", "http://localhost:6333")

mcp_registry = MovieSearchServiceRegistry.from_environment()
mcp_server = create_mcp_server(mcp_registry, enable_experimental=False)
llm_registry = LLMProviderRegistry(catalog)


async def run_movie_query(
    query: str,
    *,
    enable_critic: bool = ENABLE_CRITIC,
    rewrite_strategy: str = QUERY_REWRITE_STRATEGY,
):
    async with Client(mcp_server, raise_exceptions=True) as client:
        gateway = MCPMovieToolGateway(client)
        workflow = build_movie_assistant_workflow(
            gateway=gateway,
            catalog=catalog,
            workflow_id=WORKFLOW_ID,
            provider_registry=llm_registry,
            role_profile_overrides=ROLE_PROFILE_OVERRIDES,
            enable_critic=enable_critic,
            query_rewrite_strategy=rewrite_strategy,
        )
        return await workflow.run(query)


state = await run_movie_query(USER_QUERY)
print({"turn_id": state.turn_id, "status": state.status})

[09/16/26 15:31:22] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330222;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330223;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

                    INFO     HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"              ]8;id=10330230;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330231;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\

[09/16/26 15:31:24] INFO     HTTP Request: GET                                                      ]8;id=10330236;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330237;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/exis                
                             ts "HTTP/1.1 200 OK"                                                                  

                    INFO     HTTP Request: GET                                                      ]8;id=10330242;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330243;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1                     
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: POST                                                     ]8;id=10330248;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330249;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/count "HTTP/1.1 200 OK"                                                            

                    INFO     HTTP Request: POST                                                     ]8;id=10330254;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330255;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/count "HTTP/1.1 200 OK"                                                            

... (15 outputs left out)

[09/16/26 15:31:45] INFO     HTTP Request: POST                                                     ]8;id=10330330;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330331;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/query "HTTP/1.1 200 OK"                                                            

                    INFO     HTTP Request: POST                                                     ]8;id=10330336;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330337;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts "HTTP/1.1 200 OK"                                                                  

                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330342;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330343;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

{'turn_id': 'b0ef2b3d-f5d5-4db2-9114-fae22ab8dbae', 'status': 'completed'}


## 8. Ispezione del risultato

Non guardiamo soltanto il testo finale. Per capire il comportamento del sistema osserviamo piano, query realmente inviata a MCP, ranking, filtri e risposta strutturata.

In [8]:
display(Markdown("### Piano del Planner"))
display(state.plan.model_dump())

search = state.latest_search
display(Markdown("### Risposta del tool `search_movies`"))
print({
    "status": search.status,
    "query_inviata": search.query,
    "filtri_applicati": search.applied_filters.model_dump(
        exclude_none=True, exclude_defaults=True
    ),
    "profilo_retrieval": search.pipeline.profile_id,
    "numero_risultati": search.returned_count,
})

result_rows = [
    {
        "rank": item.rank,
        "movie_id": item.movie_id,
        "title": item.title,
        "dense_score": item.scores.dense,
        "year": item.year,
        "runtime": item.runtime_minutes,
        "genres": " | ".join(item.genres),
    }
    for item in search.results
]
display(pd.DataFrame(result_rows))

display(Markdown("### Risposta finale grounded"))
display(Markdown(state.final_answer.answer))
display(pd.DataFrame([r.model_dump() for r in state.final_answer.recommendations]))

### Piano del Planner

{'action': 'search',
 'detected_language': 'it',
 'semantic_query': "un film malinconico sulla memoria e l'identità",
 'semantic_query_language': 'it',
 'rewrite_strategy': 'none',
 'filters': {'min_runtime': None,
  'max_runtime': 120.0,
  'min_year': None,
  'max_year': None,
  'languages': [],
  'countries': [],
  'include_genres': [],
  'exclude_genres': ['Horror'],
  'adult': None},
 'ambiguities': [],
 'clarification_question': None,
 'reason': 'The user is looking for a melancholic movie about memory and identity, excluding horror, with a runtime under two hours. All constraints map cleanly to semantic query and hard filters.'}

### Risposta del tool `search_movies`

{'status': 'ok', 'query_inviata': "un film malinconico sulla memoria e l'identità", 'filtri_applicati': {'max_runtime': 120.0, 'exclude_genres': ['Horror']}, 'profilo_retrieval': 'dense_v1', 'numero_risultati': 10}


,rank,movie_id,title,dense_score,year,runtime,genres
0,1,4226,Memento (2000),0.530574,2000,113,Mystery | Thriller
1,2,1270,Back to the Future (1985),0.471600,1985,116,Adventure | Comedy | Sci-Fi | Science Fiction
2,3,367,"Mask, The (1994)",0.464471,1994,101,Action | Comedy | Crime | Fantasy | Romance
3,4,608,Fargo (1996),0.451931,1996,98,Comedy | Crime | Drama | Thriller
4,5,1580,Men in Black (a.k.a. MIB) (1997),0.446176,1997,98,Action | Comedy | Sci-Fi | Adventure | Science...
5,6,50,"Usual Suspects, The (1995)",0.418508,1995,106,Crime | Mystery | Thriller | Drama
6,7,1198,Raiders of the Lost Ark (Indiana Jones and the...,0.417392,1981,115,Action | Adventure
7,8,4306,Shrek (2001),0.416087,2001,90,Adventure | Animation | Children | Comedy | Fa...
8,9,1,Toy Story (1995),0.414327,1995,81,Adventure | Animation | Children | Comedy | Fa...
9,10,377,Speed (1994),0.407649,1994,116,Action | Romance | Thriller


### Risposta finale grounded

In base ai tuoi criteri, ti consiglio "Memento (2000)". Questo thriller psicologico e misterioso diretto da Christopher Nolan ha una durata di 113 minuti (sotto le due ore) e non appartiene al genere horror. La pellicola esplora profondamente i temi della memoria e dell'identità attraverso la storia di Leonard Shelby, un uomo che soffre di una rara forma di perdita di memoria a breve termine e cerca di rintracciare l'assassino di sua moglie.

,movie_id,title,explanation
0,4226,Memento (2000),Esplora i temi della memoria e dell'identità s...


## 9. Trace: modello, prompt, token e latenza

Le trace permettono di confrontare provider e configurazioni senza registrare API key o prompt completi. `search_history` conserva invece ogni tentativo di retrieval, utile quando il Critic richiede un retry.

In [9]:
llm_trace_rows = []
for trace in state.llm_traces:
    row = trace.model_dump(mode="json")
    usage = row.pop("usage")
    row.update(usage)
    llm_trace_rows.append(row)

display(Markdown("### Chiamate LLM"))
display(pd.DataFrame(llm_trace_rows))
display(Markdown("### Chiamate tool"))
display(pd.DataFrame([t.model_dump(mode="json") for t in state.tool_traces]))

### Chiamate LLM

,agent,provider,profile_id,model,prompt_version,started_at,duration_ms,request_id,input_tokens,output_tokens,total_tokens
0,planner,gemini,gemini_free,gemini-3.5-flash-lite,planner_v1,2026-09-16T13:31:22.712442Z,1853.77525,K5qqau_LJfGb3boP-9i7wQk,448,215,663
1,answer,gemini,gemini_free,gemini-3.5-flash-lite,answer_v1,2026-09-16T13:31:45.913277Z,2170.83625,QpqqaoCtO_CYvdIP6fbG4Q0,7404,239,7643


### Chiamate tool

,tool,started_at,duration_ms,status,returned_count,error_type
0,search_movies,2026-09-16T13:31:24.566470Z,21307.470750,ok,10,None
1,get_movie_details,2026-09-16T13:31:45.874074Z,38.895959,ok,5,None


## 10. No-results e filtri stretti

Una ricerca vuota non è un errore tecnico. Il workflow non chiama `get_movie_details`; l'Answer agent deve restituire `status=no_results` e può soltanto proporre all'utente di modificare esplicitamente un vincolo. Nessun filtro viene rilassato automaticamente.

In [ ]:
RUN_NO_RESULTS_EXAMPLE = False

if RUN_NO_RESULTS_EXAMPLE:
    strict_state = await run_movie_query(
        "Vorrei un film in lingua italiana, uscito prima del 1960, "
        "sotto gli 80 minuti e non horror."
    )
    print({
        "workflow_status": strict_state.status,
        "search_status": strict_state.latest_search.status,
        "details_called": strict_state.details_response is not None,
        "answer_status": strict_state.final_answer.status,
    })
    display(Markdown(strict_state.final_answer.answer))

[09/16/26 15:38:38] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330348;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330349;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.92it/s]


[09/16/26 15:38:43] INFO     HTTP Request: POST                                                     ]8;id=10330354;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330355;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/query "HTTP/1.1 200 OK"                                                            

                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330360;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330361;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

{'workflow_status': 'completed', 'search_status': 'no_results', 'details_called': False, 'answer_status': 'no_results'}


Non ho trovato alcun film nel catalogo che soddisfi tutti i criteri specificati (lingua italiana, uscita prima del 1960, durata inferiore a 80 minuti e nessun elemento horror). Potresti considerare di rilassare o chiarire alcuni vincoli per trovare dei risultati.

## 11. Confronto italiano/inglese

BGE-M3 e `bge-reranker-v2-m3` sono multilingue, ma due formulazioni o traduzioni non producono obbligatoriamente lo stesso ranking. Il blocco seguente misura l'overlap della top-10 con `query_rewrite_strategy=none`; abilitarlo comporta quattro chiamate LLM complessive (Planner + Answer per query).

In [ ]:
RUN_LANGUAGE_COMPARISON = False

if RUN_LANGUAGE_COMPARISON:
    italian_state = await run_movie_query(
        "Un film malinconico sulla memoria e l'identità"
    )
    english_state = await run_movie_query(
        "A melancholic film about memory and identity"
    )
    italian_ids = [r.movie_id for r in italian_state.latest_search.results]
    english_ids = [r.movie_id for r in english_state.latest_search.results]
    overlap = len(set(italian_ids) & set(english_ids))
    print({
        "italian_query_to_mcp": italian_state.latest_search.query,
        "english_query_to_mcp": english_state.latest_search.query,
        "overlap@10": overlap,
        "same_order": italian_ids == english_ids,
    })

[09/16/26 16:15:54] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330366;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330367;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]


[09/16/26 16:15:59] INFO     HTTP Request: POST                                                     ]8;id=10330372;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330373;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/query "HTTP/1.1 200 OK"                                                            

                    INFO     HTTP Request: POST                                                     ]8;id=10330378;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330379;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts "HTTP/1.1 200 OK"                                                                  

                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330384;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330385;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

[09/16/26 16:16:02] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330390;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330391;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.49it/s]


[09/16/26 16:16:06] INFO     HTTP Request: POST                                                     ]8;id=10330396;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330397;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/query "HTTP/1.1 200 OK"                                                            

                    INFO     HTTP Request: POST                                                     ]8;id=10330402;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330403;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts "HTTP/1.1 200 OK"                                                                  

                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330408;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330409;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

{'italian_query_to_mcp': "Un film malinconico sulla memoria e l'identità", 'english_query_to_mcp': 'A melancholic film about memory and identity', 'overlap@10': 8, 'same_order': False}


In [ ]:
display(Markdown("### Risultati della comparazione linguistica"))
display(Markdown("#### Query in italiano"))
print(italian_state.model_dump_json())
display(Markdown("#### Query in inglese"))
print(english_state.model_dump_json())   

### Risultati della comparazione linguistica

#### Query in italiano

{'contract_version': 'movie-llm/v1', 'turn_id': '3d05e4cc-8814-4a11-8ea8-a274f11e2315', 'workflow_id': 'grounded_v1', 'original_query': "Un film malinconico sulla memoria e l'identità", 'catalog_language': 'en', 'query_rewrite_strategy': 'none', 'status': 'completed', 'plan': {'action': 'search', 'detected_language': 'it', 'semantic_query': "Un film malinconico sulla memoria e l'identità", 'semantic_query_language': 'it', 'rewrite_strategy': 'none', 'filters': {'min_runtime': None, 'max_runtime': None, 'min_year': None, 'max_year': None, 'languages': [], 'countries': [], 'include_genres': [], 'exclude_genres': [], 'adult': None}, 'ambiguities': [], 'clarification_question': None, 'reason': 'The user provided a clear semantic query about melancholic movies regarding memory and identity without any explicit conflicting constraints.'}, 'search_history': [{'contract_version': 'movie-search/v1', 'status': 'ok', 'query': "Un film malinconico sulla memoria e l'identità", 'applied_filters': {'

#### Query in inglese

{'contract_version': 'movie-llm/v1', 'turn_id': 'eb6b2a35-34b6-47fa-ba25-7dc369f43400', 'workflow_id': 'grounded_v1', 'original_query': 'A melancholic film about memory and identity', 'catalog_language': 'en', 'query_rewrite_strategy': 'none', 'status': 'completed', 'plan': {'action': 'search', 'detected_language': 'en', 'semantic_query': 'A melancholic film about memory and identity', 'semantic_query_language': 'en', 'rewrite_strategy': 'none', 'filters': {'min_runtime': None, 'max_runtime': None, 'min_year': None, 'max_year': None, 'languages': [], 'countries': [], 'include_genres': [], 'exclude_genres': [], 'adult': None}, 'ambiguities': [], 'clarification_question': None, 'reason': 'The user provided a clear thematic request for a melancholic film about memory and identity without any conflicting hard constraints or ambiguities.'}, 'search_history': [{'contract_version': 'movie-search/v1', 'status': 'ok', 'query': 'A melancholic film about memory and identity', 'applied_filters': {

## 12. Abilitare il Critic come esperimento separato

Il Critic può accettare i risultati, chiedere un chiarimento o proporre una nuova query semantica. Non può cambiare i filtri. L'orchestratore riusa lo stesso `MovieFilterInput` e applica `max_retries` dal profilo.

In [37]:
RUN_CRITIC_EXPERIMENT = True

AMBIGUOUS_USER_QUERY = (
    "un film sulla guerra di unificazione italiana avvenuta nel 1800, il film può essere di qualsiasi anno"
)

if RUN_CRITIC_EXPERIMENT:
    critic_state = await run_movie_query(AMBIGUOUS_USER_QUERY, enable_critic=True)
    print(critic_state.model_dump())
    # print({
    #     "decision": critic_state.critique.decision,
    #     "reason": critic_state.critique.reason,
    #     "retry_count": critic_state.retry_count,
    #     "queries_inviate": [s.query for s in critic_state.search_history],
    #     "filters_per_tentativo": [
    #         s.applied_filters.model_dump(exclude_none=True, exclude_defaults=True)
    #         for s in critic_state.search_history
    #     ],
    # })

[09/16/26 16:44:39] INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330534;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330535;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.41it/s]


[09/16/26 16:44:43] INFO     HTTP Request: POST                                                     ]8;id=10330540;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330541;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/query "HTTP/1.1 200 OK"                                                            

                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330546;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330547;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.45it/s]


[09/16/26 16:44:46] INFO     HTTP Request: POST                                                     ]8;id=10330552;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330553;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/query "HTTP/1.1 200 OK"                                                            

                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330558;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330559;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

[09/16/26 16:44:47] INFO     HTTP Request: POST                                                     ]8;id=10330564;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=10330565;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts "HTTP/1.1 200 OK"                                                                  

                    INFO     AFC is enabled with max remote calls: 10.                               ]8;id=10330570;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py\models.py]8;;\:]8;id=10330571;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/google/genai/models.py#8416\8416]8;;\

{'contract_version': 'movie-llm/v1', 'turn_id': '63e265bf-fdcb-4026-abd3-d3c4b3d2a5f5', 'workflow_id': 'grounded_v1', 'original_query': 'un film sulla guerra di unificazione italiana avvenuta nel 1800, il film può essere di qualsiasi anno', 'catalog_language': 'en', 'query_rewrite_strategy': 'none', 'status': 'completed', 'plan': {'action': 'search', 'detected_language': 'it', 'semantic_query': 'guerra di unificazione italiana Risorgimento 1800s', 'semantic_query_language': 'it', 'rewrite_strategy': 'none', 'filters': {'min_runtime': None, 'max_runtime': None, 'min_year': None, 'max_year': None, 'languages': [], 'countries': [], 'include_genres': [], 'exclude_genres': [], 'adult': None}, 'ambiguities': [], 'clarification_question': None, 'reason': 'The user is looking for a movie about the Italian unification war in the 19th century with no year restrictions, so a semantic search for the Italian unification is appropriate.'}, 'search_history': [{'contract_version': 'movie-search/v1', '

In [38]:
print(critic_state.model_dump_json())

{"contract_version":"movie-llm/v1","turn_id":"63e265bf-fdcb-4026-abd3-d3c4b3d2a5f5","workflow_id":"grounded_v1","original_query":"un film sulla guerra di unificazione italiana avvenuta nel 1800, il film può essere di qualsiasi anno","catalog_language":"en","query_rewrite_strategy":"none","status":"completed","plan":{"action":"search","detected_language":"it","semantic_query":"guerra di unificazione italiana Risorgimento 1800s","semantic_query_language":"it","rewrite_strategy":"none","filters":{"min_runtime":null,"max_runtime":null,"min_year":null,"max_year":null,"languages":[],"countries":[],"include_genres":[],"exclude_genres":[],"adult":null},"ambiguities":[],"clarification_question":null,"reason":"The user is looking for a movie about the Italian unification war in the 19th century with no year restrictions, so a semantic search for the Italian unification is appropriate."},"search_history":[{"contract_version":"movie-search/v1","status":"ok","query":"guerra di unificazione italiana

#### NOTA: giro di clarify
ad esempio per query:
AMBIGUOUS_USER_QUERY = (
    "Non so bene che film cercare"
)

il flusso entra in clarify direttamente spinto dal planner in quanto non ci sono film disponibili dai filtri.



## 13. Passare da Gemini a OpenAI o usare modelli diversi per ruolo

Dopo aver definito `OPENAI_API_KEY` nel `.env` e installato l'extra `llm-openai`, non cambia alcuna classe. È sufficiente selezionare un profilo registrato:

```python
ROLE_PROFILE_OVERRIDES = {
    "planner": "gemini_free",
    "answer": "openai_default",
    "critic": "gemini_free",
}
```

Per cambiare nome modello, parametri o endpoint si modifica/duplica un profilo in `configs/llm_profiles_v1.json`. `OPENAI_BASE_URL` consente anche un gateway o runtime locale, purché implementi la Responses API e gli structured outputs usati dall'adapter. Non è previsto fallback automatico tra provider: un fallback renderebbe meno riproducibili costi, qualità ed errori.

## 14. Esportazione opzionale dello stato

Lo stato contiene piano, risultati, risposta e trace, ma non API key né prompt completi. Può diventare la base del dataset di valutazione end-to-end. Poiché include la richiesta utente, l'esportazione rimane esplicita.

In [ ]:
EXPORT_STATE = False

if EXPORT_STATE:
    output_dir = PROJECT_ROOT / "artifacts" / "llm_v1"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"turn_{state.turn_id}.json"
    output_path.write_text(state.model_dump_json(indent=2), encoding="utf-8")
    print(output_path)

## 15. Come evolvere verso flussi multi-agente

Le classi attuali sono già predisposte per ruoli aggiuntivi. Per introdurre un nuovo agente conviene seguire sempre questi passi:

1. definire un output Pydantic specifico;
2. creare un prompt versionato in `src/movie_rag/prompts/`;
3. collegare il ruolo a un profilo LLM nel registry;
4. aggiungere una transizione esplicita nell'orchestratore;
5. fissare condizioni di arresto e numero massimo di retry;
6. aggiungere test con provider e tool simulati;
7. confrontare qualità, token, latenza e costo con il workflow più semplice.

Esempi futuri sono un verificatore deterministico dei vincoli, un agente di elicitation multi-turn o un supervisor che seleziona workflow predefiniti. Il contratto MCP e gli adapter Gemini/OpenAI possono restare invariati.

## 16. Criteri di accettazione di questa fase

Prima di ampliare l'orchestrazione registrare almeno:

- percentuale di piani con filtri corretti;
- richieste per cui il Planner chiede chiarimento;
- titoli fuori catalogo: devono essere zero nel flusso grounded;
- rispetto dei vincoli nei risultati e nella risposta;
- differenze italiano/inglese e con/senza traduzione;
- token, latenza e costo per ruolo;
- confronto `Planner → dense → Answer` contro baseline dense e LLM closed-book;
- solo successivamente, delta qualità/costo del Critic.

La guida completa è in `docs/llm_orchestration_v1.md`.